# Essai algortihme LOOCV avec le modèle de Gradient Boosting

## 1) Import des librairies

In [43]:
import mysql.connector

In [44]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt


### Chargement de la base de données :

In [45]:
connexion = mysql.connector.connect(
    host     = 'localhost',  # adresse du serveur (WAMP = localhost)
    user     = 'root',       # utilisateur par défaut sur WAMP
    password = '',           # mot de passe (vide par défaut sur WAMP)
    database = 'gamesale'    # nom de ta base de données
)

### Unification des données et extraction des tables

In [46]:

#1 table jeux
df = pd.read_sql("""
    SELECT
        id_jeu,
        nom_jeu,
        annee,
        age_requis,
        nb_succes,
        nb_avis_pos,
        nb_avis_neg,
        temps_jeu_moyen,
        prix,
        ventes_Global
    FROM jeux
""", connexion)
df['ventes_Global'] = df['ventes_Global'].str.replace(',', '.').astype(float)

print(df)

#2 table genre 
# GROUP BY id_jeu = on garde une seule ligne par jeu (le 1er genre)
df_genre = pd.read_sql("""
    SELECT id_jeu, genre
    FROM genre
    GROUP BY id_jeu
""", connexion)

print(df_genre)

#3 Table editeur
# GROUP BY id_jeu = on garde un seul éditeur par jeu (le 1er)
df_edit = pd.read_sql("""
    SELECT id_jeu, id_editeur
    FROM editeur
    GROUP BY id_jeu
""", connexion)

print(df_edit)

#4 table developpeur
# GROUP BY id_jeu = on garde un seul développeur par jeu (le 1er)
df_dev = pd.read_sql("""
    SELECT id_jeu, id_developpeur
    FROM developpeur
    GROUP BY id_jeu
""", connexion)

print(df_dev)


#5 table a_os
# id_os : 1 = Windows, 2 = Mac, 3 = Linux
# CASE WHEN id_os = 1 THEN 1 ELSE 0 END
#   → met 1 si c'est Windows, 0 sinon
# MAX() permet de garder le 1 s'il existe parmi toutes les lignes du jeu
# GROUP BY id_jeu = une seule ligne par jeu avec les 3 colonnes OS

df_os = pd.read_sql("""
    SELECT
        id_jeu,
        MAX(CASE WHEN id_os = 1 THEN 1 ELSE 0 END) AS os_windows,
        MAX(CASE WHEN id_os = 2 THEN 1 ELSE 0 END) AS os_mac,
        MAX(CASE WHEN id_os = 3 THEN 1 ELSE 0 END) AS os_linux
    FROM a_os
    GROUP BY id_jeu
""", connexion)

print(df_os)

#6 table a_categorie
# Même logique que les OS :
# Pour chaque catégorie importante, on crée une colonne 0/1
# 1 = le jeu a cette catégorie, 0 = il ne l'a pas

df_cat = pd.read_sql("""
    SELECT
        id_jeu,
        MAX(CASE WHEN id_cat = '1'  THEN 1 ELSE 0 END) AS cat_multi,
        MAX(CASE WHEN id_cat = '2'  THEN 1 ELSE 0 END) AS cat_online,
        MAX(CASE WHEN id_cat = '4'  THEN 1 ELSE 0 END) AS cat_vac,
        MAX(CASE WHEN id_cat = '5'  THEN 1 ELSE 0 END) AS cat_solo,
        MAX(CASE WHEN id_cat = '6'  THEN 1 ELSE 0 END) AS cat_cloud,
        MAX(CASE WHEN id_cat = '7'  THEN 1 ELSE 0 END) AS cat_achiev,
        MAX(CASE WHEN id_cat = '8'  THEN 1 ELSE 0 END) AS cat_cards,
        MAX(CASE WHEN id_cat = '10' THEN 1 ELSE 0 END) AS cat_ctrl,
        MAX(CASE WHEN id_cat = '22' THEN 1 ELSE 0 END) AS cat_workshop
    FROM a_categorie
    GROUP BY id_jeu
""", connexion)

print(df_cat)


#7 table a_tag
# La table a_tag a une colonne par tag (ex: action, rpg, shooter...)
# Chaque colonne vaut 1 si le jeu a ce tag, 0 sinon
# On veut compter combien de tags chaque jeu possède au total

# Étape 1 : on récupère les noms de toutes les colonnes de la table
curseur       = connexion.cursor()
curseur.execute("SELECT * FROM a_tag LIMIT 1")
curseur.fetchall()  # nécessaire pour que .description soit disponible
toutes_cols   = [description[0] for description in curseur.description]
colonnes_tags = toutes_cols[1:]  # on enlève "id_jeu" qui est en position 0

# Étape 2 : on construit la partie SQL qui additionne tous les tags
# Résultat : "action + rpg + shooter + ..."
somme_tags = ' + '.join([f'`{col}`' for col in colonnes_tags])

# Étape 3 : on exécute la requête
df_tags = pd.read_sql(f"""
    SELECT
        id_jeu,
        ({somme_tags}) AS nb_tags
    FROM a_tag
""", connexion)

print(df_tags)

     id_jeu                       nom_jeu annee  age_requis  nb_succes  \
0         1            grand theft auto v  2015          18         77   
1         2  grand theft auto san andreas  2008          18          0   
2         3    grand theft auto vice city  2011          18          0   
3         4        call of duty black ops  2010          18         68   
4         5     call of duty black ops ii  2012          18         35   
..      ...                           ...   ...         ...        ...   
775     776               hospital tycoon  2009           0          0   
776     777              summer athletics  2009           0          0   
777     778     spore galactic adventures  2009           0          0   
778     779                       15 days  2015           0          0   
779     780                     teslagrad  2013           0         36   

     nb_avis_pos  nb_avis_neg  temps_jeu_moyen   prix  ventes_Global  
0         329061       139308           

C:\Users\Catherine\AppData\Local\Temp\ipykernel_20688\594151201.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""
C:\Users\Catherine\AppData\Local\Temp\ipykernel_20688\594151201.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_genre = pd.read_sql("""
C:\Users\Catherine\AppData\Local\Temp\ipykernel_20688\594151201.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_edit = pd.read_sql("""
C:\Users\Catherine\AppData\Local\Temp\ipykernel_20688\594151201.py:42: UserWarning: pandas only supports 

### Fusion de toutes les tables

In [47]:

# merge() fusionne deux tableaux sur une colonne commune (ici id_jeu)
# how='left' = on garde tous les jeux de df même si une info manque

df = df.merge(df_genre, on='id_jeu', how='left')
df = df.merge(df_edit,  on='id_jeu', how='left')
df = df.merge(df_dev,   on='id_jeu', how='left')
df = df.merge(df_os,    on='id_jeu', how='left')
df = df.merge(df_cat,   on='id_jeu', how='left')
df = df.merge(df_tags,  on='id_jeu', how='left')

print(f"Fusion terminée : {len(df)} jeux | {len(df.columns)} colonnes")

# On ferme la connexion à la base de données
connexion.close()
print(df)



Fusion terminée : 780 jeux | 26 colonnes
     id_jeu                       nom_jeu annee  age_requis  nb_succes  \
0         1            grand theft auto v  2015          18         77   
1         2  grand theft auto san andreas  2008          18          0   
2         3    grand theft auto vice city  2011          18          0   
3         4        call of duty black ops  2010          18         68   
4         5     call of duty black ops ii  2012          18         35   
..      ...                           ...   ...         ...        ...   
775     776               hospital tycoon  2009           0          0   
776     777              summer athletics  2009           0          0   
777     778     spore galactic adventures  2009           0          0   
778     779                       15 days  2015           0          0   
779     780                     teslagrad  2013           0         36   

     nb_avis_pos  nb_avis_neg  temps_jeu_moyen   prix  ventes_Global  

### COnfiguration des données POUR le modèle 

In [48]:

# Préparation des données pour le modèle


# Le genre est un texte ('Action', 'RPG'...) mais le modèle a besoin de nombres -x on transforme chaque genre en un nombre
# Exemple : Action=0, Adventure=1, RPG=2, Shooter=3...
encodeur = LabelEncoder()#fonction qui s'en occupe donc
df['genre']  = df['genre'].fillna('Unknown')        # si un jeu n'a pas de genre, on met 'Unknown'
df['genre_enc'] = encodeur.fit_transform(df['genre'])  # on crée la colonne numérique

# On remplace les valeurs manquantes par -1 pour éditeur et développeur
df['id_editeur']     = df['id_editeur'].fillna(-1).astype(int)
df['id_developpeur'] = df['id_developpeur'].fillna(-1).astype(int)

# FEATURES = la liste des colonnes qu'on donne au modèle pour apprendre
# Ce sont toutes les informations disponibles sur chaque jeu
FEATURES = [
    'age_requis',
    'nb_succes',
    'temps_jeu_moyen',
    'prix',
    'nb_avis_pos',
    'nb_avis_neg',
    'genre_enc',
    'id_editeur',
    'id_developpeur',
    'os_windows',
    'os_mac',
    'os_linux',
    'cat_multi',
    'cat_online',
    'cat_vac',
    'cat_solo',
    'cat_cloud',
    'cat_achiev',
    'cat_cards',
    'cat_ctrl',
    'cat_workshop',
    'nb_tags',
]

# X = le tableau de toutes les features (ce qu'on donne au modèle)
# .fillna(0) remplace les éventuelles valeurs manquantes par 0
# .values convertit le DataFrame en tableau numpy (format requis par sklearn)
X = df[FEATURES].fillna(0).values

# y = ce qu'on veut prédire : les ventes mondiales de chaque jeu
y = df['ventes_Global'].values

print(f"X : {X.shape[0]} jeux, {X.shape[1]} features")
print(f"y : {len(y)} valeurs de ventes à prédire")

X : 780 jeux, 22 features
y : 780 valeurs de ventes à prédire


### Modèle Gradient Boosting 

In [49]:
"""
C'est un modèle combinant plusieurs autres modèles. DOnc plus performant.
Sa limite est qu'il ets plus lent puisqu'il fait tourner "plusieurs auters modèles", mais étént donée que notre jeu comporte moins de  100 données
il est cohérentpoour nosu de l'utiliser
"""
model = GradientBoostingRegressor()   

#otpuna varier les paramètre
                            

## Configuration LOOCV

In [ ]:
# LOOCV = Leave One Out Cross Validation
# Principe : on a par exemple 100 jeux
#   - Tour 1  : on entraîne sur les jeux 2 à 100, on prédit le jeu 1
#   - Tour 2  : on entraîne sur les jeux 1 et 3 à 100, on prédit le jeu 2
#   - ...
#   - Tour 100 : on entraîne sur les jeux 1 à 99, on prédit le jeu 100
# À la fin, chaque jeu a été prédit exactement une fois, sans jamais
# avoir été utilisé pour l'entraînement de sa propre prédiction




loo = LeaveOneOut()

liste_valeurs_reelles=[]
liste_valeurs_predites=[]


#Boucle LOOCV
nombre_total_tours=len(X)
tour_actuel=0

# la fonction ci dessous a été entièrement toruvée sur internet, mais j'ai pris le temps de bien la comprendre.
for train_idx, test_idx in loo.split(X):
    # loo.split(X) génère à chaque tour :
    #   - train_idx : les indices des jeux utilisés pour l'entraînement
    #   - test_idx  : l'indice du jeu qu'on veut prédire (toujours 1 seul)
    tour_actuel=tour_actuel+1
    
     # On découpe X et y en données d'entraînement et de test
    X_train = X[train_idx]   # toutes les lignes sauf une
    X_test  = X[test_idx]    # la ligne qu'on veut prédire

    y_train = y[train_idx]   # les vraies ventes pour l'entraînement
    y_test  = y[test_idx]    # la vraie vente du jeu à prédire

        # ── Normalisation ──
    # On met toutes les features à la même échelle (moyenne=0, écart-type=1)

    scaler      = StandardScaler()
    X_train_normalise = scaler.fit_transform(X_train)  # calcule ET applique
    X_test_normalise  = scaler.transform(X_test)       # applique seulement


    #Entrainement
    # On entraîne le modèle (ici Gradient Boosting qui est dans le chunk précédent) sur les données d'entraînement normalisées
    model.fit(X_train_normalise, y_train)

    #Prédiction 
    # On prédit les ventes du jeu de test
    # [0] car predict() retourne une liste, on veut juste le premier élément
    valeur_predite = model.predict(X_test_normalise)[0]
    valeur_reelle  = y_test[0]

    # On stocke les résultats
    liste_valeurs_predites.append(valeur_predite)
    liste_valeurs_reelles.append(valeur_reelle)





### Calculs des métriques 

In [51]:
y_pred = np.array(liste_valeurs_predites)
y_true = np.array(liste_valeurs_reelles)


# RMSE = Root Mean Squared Error ====> erreur typique du modèle en millions d'exemplaires
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# MAE = Mean Absolute Error =====> erreur absolue moyenne en millions d'exemplaires
mae = mean_absolute_error(y_true, y_pred)

# R² = coefficient de détermination===> entre 0 et 1, indique la part de variance expliquée par le modèle
r2 = r2_score(y_true, y_pred)



# Résultats

In [52]:
print(f"\nRésultats LOOCV — Gradient Boosting")
print(f"  RMSE : {rmse:.3f} M  → en moyenne le modèle se trompe de + ou - {rmse:.2f} millions d'exemplaires")
print(f"  MAE  : {mae:.3f} M  → erreur absolue moyenne")
print(f"  R²   : {r2:.3f}   → le modèle explique {r2*100:.1f}% de la variance des ventes")


Résultats LOOCV — Gradient Boosting
  RMSE : 2.850 M  → en moyenne le modèle se trompe de + ou - 2.85 millions d'exemplaires
  MAE  : 1.303 M  → erreur absolue moyenne
  R²   : 0.438   → le modèle explique 43.8% de la variance des ventes
